# Hierarchical Stackelberg Security Problem

This is the authoritative implementation notebook. It follows the fixed 12-phase architecture and governing Phase 0 contracts. The Attacker best response is computed by one of two selectable Bellman formulations: the preserved fixed-time snapped lattice or the physical successor-grid graph with virtual switching states. No CasADi/IPOPT NLP is part of the authoritative pipeline.

## Phase 1 — Project Initialization and Configuration

**Responsibility:** load the centralized configuration, create standard project paths, initialize logging, and validate configuration consistency.

This phase performs no terrain construction, LOS geometry, symbolic modeling, cost-map computation, optimization, export, or plotting.

In [1]:
from pathlib import Path
import sys

project_root = Path.cwd()

if project_root.name == "p1b_4D":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from p1b_4D.configuration import build_configuration_bundle

configuration_bundle = build_configuration_bundle()

# One authoritative switch; both solver implementations remain available.
# ATTACKER_TRANSITION_MODEL = "snapped_fixed_time_step" # Deprecated
ATTACKER_TRANSITION_MODEL = "successor_grid_physical_edge"
configuration_bundle["primary_result"]["attacker_solver_config"]["transition_model"] = ATTACKER_TRANSITION_MODEL

if not configuration_bundle["status"]["success"]:
    raise RuntimeError(configuration_bundle["status"]["message"])

configuration_bundle["validation"]["summary"]

2026-07-28 16:50:55,220 | INFO | stackelberg | phase=Phase 1: Configuration status=started
2026-07-28 16:50:55,221 | WARNING | stackelberg | phase=Phase 1: Configuration warning=Vehicle segment_length is deferred to the future transcription contract instead of fabricating a Phase 1 value
2026-07-28 16:50:55,222 | INFO | stackelberg | phase=Phase 1: Configuration status=success elapsed_seconds=0.001127


'Phase 1 configuration is valid'

## Phase 2 — Terrain Model

**Responsibility:** construct the authoritative terrain model, terrain derivatives and samples, terrain-following sensor position, fixed goal, sensor-dependent LOS tangent/boundary/masks, and LOS coverage area. Results are validated and exported without plotting.

In [2]:
from p1b_4D.geometry import build_geometry_bundle

phase_logger = configuration_bundle["primary_result"]["logging_utilities"]["logger"]
phase_context = configuration_bundle["primary_result"]["logging_utilities"]["phase_context"]
with phase_context(phase_logger, "Phase 2: Terrain and LOS Geometry") as phase_status:
    geometry_bundle = build_geometry_bundle(configuration_bundle)
    phase_status["warnings"].extend(geometry_bundle["status"]["warnings"])
    if not geometry_bundle["status"]["success"]:
        raise RuntimeError(geometry_bundle["status"]["message"])
geometry_bundle["validation"]["summary"]

2026-07-28 16:50:55,543 | INFO | stackelberg | phase=Phase 2: Terrain and LOS Geometry status=started
2026-07-28 16:50:55,545 | INFO | stackelberg | phase=Phase 2: Terrain and LOS Geometry status=success elapsed_seconds=0.002173


'Phase 2 geometry validation passed'

## Phase 3 — Sensor Geometry

Completed by the combined Phase 2 Geometry Bundle, which contains terrain-independent sensor placement, LOS tangent/boundary, masks, and coverage outputs.

## Phase 4 — CasADi Symbolic Detection Model

**Responsibility:** construct the authoritative CasADi symbolic range, LOS, powered acoustic, glide radar/radial-velocity/RCS, mission detection, mission time, normalization, and objective-component functions. Geometry is consumed from the Phase 2 Geometry Bundle and is not reconstructed.

In [3]:
from p1b_4D.detection import build_symbolic_detection_bundle

with phase_context(phase_logger, "Phase 3: CasADi Symbolic Detection") as phase_status:
    detection_bundle = build_symbolic_detection_bundle(
        configuration_bundle, geometry_bundle
    )
    phase_status["warnings"].extend(detection_bundle["status"]["warnings"])
    if not detection_bundle["status"]["success"]:
        raise RuntimeError(detection_bundle["status"]["message"])
detection_bundle["validation"]["summary"]

2026-07-28 16:50:55,563 | INFO | stackelberg | phase=Phase 3: CasADi Symbolic Detection status=started
2026-07-28 16:50:55,566 | INFO | stackelberg | phase=Phase 3: CasADi Symbolic Detection status=success elapsed_seconds=0.002729


'Phase 3 symbolic detection validation passed'

## Phase 5 — 4D Stage Cost Construction

**Responsibility:** construct the standard \(z,h,v,\gamma\) grids, state-validity masks, powered/glide detection and time components, normalized components, and the authoritative local glide \(J4D\). Invalid states receive positive-infinite cost. This phase performs no Bellman propagation or cost-to-go computation.

In [4]:
from p1b_4D.stage_cost import construct_stage_cost_4d

with phase_context(phase_logger, "Phase 4: 4D Stage Cost") as phase_status:
    stage_cost_4d_bundle = construct_stage_cost_4d(
        configuration_bundle, geometry_bundle, detection_bundle
    )
    phase_status["warnings"].extend(stage_cost_4d_bundle["status"]["warnings"])
    if not stage_cost_4d_bundle["status"]["success"]:
        raise RuntimeError(stage_cost_4d_bundle["status"]["message"])
stage_cost_4d_bundle["validation"]["summary"]

2026-07-28 16:50:55,574 | INFO | stackelberg | phase=Phase 4: 4D Stage Cost status=started
2026-07-28 16:51:08,682 | INFO | stackelberg | phase=Phase 4: 4D Stage Cost status=success elapsed_seconds=13.108058


'Phase 4 4D stage-cost validation passed'

## Phase 6 — 2D Projection

**Responsibility:** project the authoritative local \(J4D\) over feasible \(v,\gamma\) values and store diagnostic local controls. This result is visualization-only and is never a Bellman policy, value function, cost-to-go map, or trajectory source.

In [5]:
from p1b_4D.projection import construct_projected_cost_map

with phase_context(phase_logger, "Phase 5: 2D Projected Cost") as phase_status:
    projected_cost_bundle = construct_projected_cost_map(
        configuration_bundle,
        geometry_bundle,
        detection_bundle,
        stage_cost_4d_bundle,
    )
    phase_status["warnings"].extend(projected_cost_bundle["status"]["warnings"])
    if not projected_cost_bundle["status"]["success"]:
        raise RuntimeError(projected_cost_bundle["status"]["message"])
projected_cost_bundle["validation"]["summary"]

2026-07-28 16:51:08,691 | INFO | stackelberg | phase=Phase 5: 2D Projected Cost status=started
2026-07-28 16:51:08,722 | INFO | stackelberg | phase=Phase 5: 2D Projected Cost status=success elapsed_seconds=0.031145


'Phase 5 2D projected-cost validation passed'

## Phase 7 — Selectable Bellman Follower

**Responsibility:** run the Bellman formulation selected by `ATTACKER_TRANSITION_MODEL`. `snapped_fixed_time_step` preserves the original `J4D` lattice. `successor_grid_physical_edge` constructs exact grid-to-grid constant-speed edges, derives each edge angle and duration, and connects the analytic LOS switching point through a virtual first edge. Both solve a finite forward DAG exactly; only the successor-grid formulation guarantees that its piecewise-kinematic edge endpoints coincide with the reported path nodes.

In [6]:
from p1b_4D.bellman import generate_bellman_candidates
from p1b_4D.successor_grid_solver import solve_successor_grid_attacker

with phase_context(phase_logger, "Phase 6: Multi-start Coarse Bellman") as phase_status:
    if ATTACKER_TRANSITION_MODEL == "snapped_fixed_time_step":
        bellman_candidate_bundle = generate_bellman_candidates(
            configuration_bundle, geometry_bundle, detection_bundle,
            stage_cost_4d_bundle, projected_cost_bundle,
        )
        bellman_response_bundle = None
    elif ATTACKER_TRANSITION_MODEL == "successor_grid_physical_edge":
        bellman_candidate_bundle, bellman_response_bundle = solve_successor_grid_attacker(
            configuration_bundle, geometry_bundle, detection_bundle,
        )
    else:
        raise ValueError(f"Unsupported ATTACKER_TRANSITION_MODEL: {ATTACKER_TRANSITION_MODEL}")
    phase_status["warnings"].extend(bellman_candidate_bundle["status"]["warnings"])
    if not bellman_candidate_bundle["status"]["success"]:
        raise RuntimeError(bellman_candidate_bundle["status"]["message"])
bellman_candidate_bundle["validation"]["summary"]

2026-07-28 16:51:08,731 | INFO | stackelberg | phase=Phase 6: Multi-start Coarse Bellman status=started
2026-07-28 16:51:18,746 | INFO | stackelberg | phase=Phase 6: Multi-start Coarse Bellman status=success elapsed_seconds=10.014236


'Physical successor-grid candidates validated'

## Phase 8 — Selected Bellman-Optimal Attacker Response

**Responsibility:** expose the minimum-cost response from the selected finite Bellman formulation. The legacy formulation selects from snapped fixed-step candidates here; the successor-grid formulation already includes its virtual-switch minimization in Phase 7.

This is the sole authoritative Attacker best response. No CasADi/IPOPT NLP is used: `attacker_nlp.py` is retained only as a disconnected, deprecated experimental module for offline discretization-error comparison and is never called by this notebook. "Optimal" here means optimal on the discretized dynamic-programming formulation — not a claim of continuous global optimality.

In [7]:
from p1b_4D.bellman import select_authoritative_bellman_response

with phase_context(phase_logger, "Phase 7: Bellman Optimal Attacker Response") as phase_status:
    if bellman_response_bundle is None:
        bellman_response_bundle = select_authoritative_bellman_response(
            bellman_candidate_bundle, configuration_bundle,
        )
    phase_status["warnings"].extend(bellman_response_bundle["status"]["warnings"])
    if not bellman_response_bundle["status"]["success"]:
        raise RuntimeError(bellman_response_bundle["status"]["message"])
bellman_response_bundle["validation"]["summary"]

2026-07-28 16:51:18,755 | INFO | stackelberg | phase=Phase 7: Bellman Optimal Attacker Response status=started
2026-07-28 16:51:18,756 | INFO | stackelberg | phase=Phase 7: Bellman Optimal Attacker Response status=success elapsed_seconds=0.000727


'Authoritative physical successor-grid response validation passed'

## Phase 9 — Continuous Defender Optimization

**Responsibility:** expose a continuous `z_sensor` black-box evaluation and an algorithm-independent optimizer callback contract. Every callback evaluation rebuilds geometry and re-solves Bellman from scratch, selecting the Bellman-optimal Attacker response; sensor height remains `terrain(z_sensor) + mount_height`.

**Stage 1 upgrade (asymptotically-global outer search):** `hierarchical_coarse_to_fine_optimizer` (fixed coarse sample + local Brent refinement) offers no guarantee against missing a basin its coarse sample happened to skip over. This cell also runs `direct_global_optimizer` (SciPy's DIRECT algorithm, `locally_biased=False`), which adaptively partitions `[z_sensor_min, z_sensor_max]` and is provably convergent to the global optimum for Lipschitz-continuous objectives as its evaluation budget grows. Both are run and compared below; the DIRECT result becomes the authoritative `stackelberg_solution_bundle` used by every later phase.

In [ ]:
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

from p1b_4D.stackelberg_solver import (
    build_defender_optimizer_interface,
    direct_global_optimizer,
    evaluate_defender_position,
    hierarchical_coarse_to_fine_optimizer,
    solve_stackelberg_game,
)

with phase_context(phase_logger, "Phase 8: Continuous Defender Interface") as phase_status:
    defender_optimizer_interface = build_defender_optimizer_interface(
        configuration_bundle
    )
    phase_status["warnings"].extend(defender_optimizer_interface["status"]["warnings"])
    if not defender_optimizer_interface["status"]["success"]:
        raise RuntimeError(defender_optimizer_interface["status"]["message"])

    heuristic_solution_bundle = solve_stackelberg_game(
        configuration_bundle, hierarchical_coarse_to_fine_optimizer
    )
    phase_status["warnings"].extend(heuristic_solution_bundle["status"]["warnings"])
    if not heuristic_solution_bundle["status"]["success"]:
        raise RuntimeError(heuristic_solution_bundle["status"]["message"])

    stackelberg_solution_bundle = solve_stackelberg_game(
        configuration_bundle,
        lambda evaluate, bounds, options: direct_global_optimizer(
            evaluate, bounds, {**options, "direct_maxfun": 120, "direct_maxiter": 300}
        ),
    )
    phase_status["warnings"].extend(stackelberg_solution_bundle["status"]["warnings"])
    if not stackelberg_solution_bundle["status"]["success"]:
        raise RuntimeError(stackelberg_solution_bundle["status"]["message"])

heuristic_final = heuristic_solution_bundle["primary_result"]["final_stackelberg_solution"]
direct_final = stackelberg_solution_bundle["primary_result"]["final_stackelberg_solution"]
heuristic_trace = heuristic_solution_bundle["primary_result"]["outer_evaluation_summaries"]
direct_trace = stackelberg_solution_bundle["primary_result"]["outer_evaluation_summaries"]

print(
    "Heuristic (coarse-to-fine + Brent): "
    f"z_sensor={heuristic_final['optimal_z_sensor']:.3f}  "
    f"defender_objective={heuristic_final['defender_objective']:.6f}  "
    f"evaluations={len(heuristic_trace)}"
)
print(
    "DIRECT (asymptotically global):          "
    f"z_sensor={direct_final['optimal_z_sensor']:.3f}  "
    f"defender_objective={direct_final['defender_objective']:.6f}  "
    f"evaluations={len(direct_trace)}"
)
basin_agreement = (
    "SAME basin"
    if abs(heuristic_final["optimal_z_sensor"] - direct_final["optimal_z_sensor"]) < 25.0
    else "DIFFERENT basin"
)
print(
    f"Agreement: {basin_agreement}, "
    f"objective delta (DIRECT - heuristic) = "
    f"{direct_final['defender_objective'] - heuristic_final['defender_objective']:.6f}"
)

figure, axis = plt.subplots(figsize=(10.0, 6.0), constrained_layout=True)
axis.scatter(
    [item["z_sensor"] for item in heuristic_trace],
    [item["defender_objective"] for item in heuristic_trace],
    color="tab:orange", alpha=0.6, s=40,
    label=f"Heuristic evaluations (n={len(heuristic_trace)})", zorder=10,
)
axis.scatter(
    [item["z_sensor"] for item in direct_trace],
    [item["defender_objective"] for item in direct_trace],
    color="tab:blue", alpha=0.5, s=40, marker="x",
    label=f"DIRECT evaluations (n={len(direct_trace)})", zorder=11,
)
axis.axvline(
    heuristic_final["optimal_z_sensor"], color="tab:orange", linestyle="--",
    linewidth=2.0, label=f"Heuristic choice z={heuristic_final['optimal_z_sensor']:.1f}",
)
axis.axvline(
    direct_final["optimal_z_sensor"], color="tab:blue", linestyle="--",
    linewidth=2.0, label=f"DIRECT choice z={direct_final['optimal_z_sensor']:.1f}",
)
axis.set_xlabel("Sensor position z_sensor [m]")
axis.set_ylabel("Defender objective")
axis.set_title("Stage 1: heuristic coarse-to-fine vs. asymptotically-global DIRECT search")
axis.grid(True, linewidth=0.4, alpha=0.3)
handles, labels = axis.get_legend_handles_labels()
unique = dict(zip(labels, handles))
axis.legend(unique.values(), unique.keys(), loc="upper left", bbox_to_anchor=(0.0, -0.14), ncol=2, frameon=True)

diagnostic_figure_path = (
    configuration_bundle["primary_result"]["project_paths"].figure_dir
    / "figure_diagnostic_stage1_outer_search.png"
)
figure.savefig(diagnostic_figure_path, dpi=150, bbox_inches="tight")
plt.close(figure)
print(f"Saved comparison figure to {diagnostic_figure_path}")

stackelberg_solution_bundle["validation"]["summary"]

2026-07-28 16:51:19,035 | INFO | stackelberg | phase=Phase 8: Continuous Defender Interface status=started


## Phase 10 — Stackelberg Solver

`solve_stackelberg_game(configuration_bundle, optimizer)` executes whichever injected `DefenderOptimizer` is supplied; both the legacy heuristic and the asymptotically-global DIRECT search share the same nested-evaluation machinery (Bellman is re-solved from scratch for every candidate `z_sensor`, for both optimizers). The comparison plot in Phase 9 shows where each optimizer sampled and which final `z_sensor` each one selected — `DIFFERENT basin` would mean the heuristic missed a better Defender placement that DIRECT found; `SAME basin` means the heuristic's answer now has independent corroboration rather than being merely plausible. Every later phase (Export, Visualization) consumes the DIRECT-based `stackelberg_solution_bundle`.

## Phase 11 — Export

**Responsibility:** perform all active disk writes through the single standardized exporter. Every computational bundle (geometry, detection, stage cost, projected cost, Bellman candidates, the Bellman-optimal Attacker response, and the Stackelberg solution) is written as JSON metadata plus NPZ arrays.

In [ ]:
from p1b_4D.result_export import export_all_results

with phase_context(phase_logger, "Phase 9: Standardized Result Export") as phase_status:
    result_export_status = export_all_results(
        configuration_bundle,
        geometry_bundle,
        detection_bundle,
        stage_cost_4d_bundle,
        projected_cost_bundle,
        bellman_candidate_bundle,
        bellman_response_bundle,
        stackelberg_solution_bundle,
    )
    phase_status["warnings"].extend(result_export_status["status"]["warnings"])
result_export_status["primary_result"]["export_status"]

2026-07-23 19:05:02,416 | INFO | stackelberg | phase=Phase 9: Standardized Result Export status=started


2026-07-23 19:05:04,418 | INFO | stackelberg | phase=Phase 9: Standardized Result Export status=success elapsed_seconds=2.002239


'complete'

## Phase 12 — Visualization

**Responsibility:** load standardized JSON/NPZ exports and render all five publication figures without calling geometry, cost, Bellman, or Defender computation modules.

In [ ]:
from p1b_4D.result_import import import_result_collection
from p1b_4D.visualization import generate_project_visualizations

with phase_context(phase_logger, "Phase 10: Visualization") as phase_status:
    imported_result_collection = import_result_collection(
        result_export_status["primary_result"]["master_manifest_path"]
    )
    visualization_result = generate_project_visualizations(
        imported_result_collection,
        configuration_bundle["primary_result"]["project_paths"].figure_dir,
    )
    phase_status["warnings"].extend(visualization_result["status"]["warnings"])
visualization_result["primary_result"]["generated_figures"]

2026-07-23 19:05:04,424 | INFO | stackelberg | phase=Phase 10: Visualization status=started


2026-07-23 19:05:14,411 | INFO | stackelberg | phase=Phase 10: Visualization status=success elapsed_seconds=9.986192


('figure_1_geometry_overview',
 'figure_2_projected_cost',
 'figure_3_cost_to_go',
 'figure_4_all_paths',
 'figure_5_stackelberg_solution')

### Visualization implementation status

All five figures are generated exclusively from the complete standardized exported collection. The Attacker path shown in every figure is the Bellman-optimal discrete trajectory; no CasADi/IPOPT NLP output is plotted.